In [1]:
!apt install swig cmake ffmpeg xvfb python3-opengl

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.15).
Suggested packages:
  libgle3 python3-numpy python3-tk swig-doc swig-examples swig4.0-examples
  swig4.0-doc
The following NEW packages will be installed:
  freeglut3 libglu1-mesa python3-opengl swig swig4.0
0 upgraded, 5 newly installed, 0 to remove and 35 not upgraded.
Need to get 1,940 kB of archives.
After this operation, 13.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 freeglut3 amd64 2.8.1-6 [74.0 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglu1-mesa amd64 9.0.2-1 [145 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 python3-opengl all 3.1.5+dfsg-1 [605 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/u

In [11]:
import os
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

%env MUJOCO_GL=egl

!pip install pyvirtualdisplay imageio[ffmpeg]

from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

env: MUJOCO_GL=egl


In [21]:
# Prepare to load data from google drive
from google.colab import drive
import datetime

# CONNECT TO GOOGLE DRIVE
gdrive_path = '/content/drive'
drive.mount(gdrive_path)

# DEFINE WORK DIRECTORY
current_step = 'step_003'
# workDir = f'{gdrive_path}/My Drive/Research/{current_step}'
workDir = os.path.join(gdrive_path, 'My Drive', 'Research', current_step)
print('WorkDir:', workDir)

log_dir = os.path.join(workDir, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
print('LogDir:', log_dir)

# create folder if it doesn't exists
if not os.path.exists(log_dir):
  os.makedirs(log_dir)

tf_log_dir = os.path.join(workDir, 'tf_logs')
print('TfLogDir:', tf_log_dir)

# create folder if it doesn't exists
if not os.path.exists(tf_log_dir):
  os.makedirs(tf_log_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
WorkDir: /content/drive/My Drive/Research/step_003
LogDir: /content/drive/My Drive/Research/step_003/20250831-224555
TfLogDir: /content/drive/My Drive/Research/step_003/tf_logs


In [6]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

CUDA available: True
Device name: Tesla T4


In [ ]:
import multiprocessing
print(multiprocessing.cpu_count())
print(os.cpu_count())
print(len(os.sched_getaffinity(0)))

In [8]:
# clean content folder
import os
import shutil

# location
location = "/content"

# directories
dirs = ["sample_data", "rl-zoo", "gym_darwin_op3", "videos"]

for dir in dirs:
    path = os.path.join(location, dir)
    try:
        shutil.rmtree(path)
    except OSError as e:
        print("Error: %s : %s" % (path, e.strerror))

Error: /content/rl-zoo : No such file or directory
Error: /content/gym_darwin_op3 : No such file or directory
Error: /content/videos : No such file or directory


In [ ]:
# Install Darwin Model
model_path = '/content/gym_darwin_op3'

if os.path.isdir(model_path):
  print(f"The directory '{model_path}' exists - git pull")
  %cd {model_path}
  !git pull
  %cd /
else:
  print(f"The directory '{model_path}' does not exist - git clone")
  !git clone --single-branch --branch {current_step} https://github.com/Gianzanti/robofei_mestrado.git {model_path}


In [ ]:
!pip install -e {model_path}

In [ ]:
# Install RL Zoo
trainner_path = '/content/rl-zoo'

if os.path.isdir(trainner_path):
  print(f"The directory '{trainner_path}' exists - git pull")
  %cd {trainner_path}
  !git pull
  %cd /
else:
  print(f"The directory '{trainner_path}' does not exist - git clone")
  !git clone https://github.com/Gianzanti/rl-zoo.git {trainner_path}


In [ ]:
!pip install -e {trainner_path}

In [ ]:
path = os.path.join(trainner_path, "logs")
if os.path.isdir(path):
  shutil.rmtree(path)

path = os.path.join(trainner_path, "research_logs/DarwinOp3-v2")
if os.path.isdir(path):
  shutil.rmtree(path)

In [22]:
# Hyper Parameters Tunning
%cd {trainner_path}

# algos_cpu = ['ppo', 'a2c']
algos_cpu = ['a2c']
algos_cuda = ['ddpg', 'sac', 'td3']

n_timestep = 100_000
save_freq = int(n_timestep / 4)
eval_freq = int(n_timestep / 4)

max_episode_steps = 1000
wrapper = [{"gymnasium.wrappers.TimeLimit": {"max_episode_steps": max_episode_steps}}]

for algo in algos_cpu:
  print('Training:', algo)
  config = f'research_config/{algo}.yml'

  !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
    --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
    --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
    --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 \
    --hyperparams n_timesteps:{n_timestep} env_wrapper:"{wrapper}"\
    --device cpu
  # forward_velocity_weight:5.0 ctrl_cost_weight:1e-3
  !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n {max_episode_steps} \
    --load-best -o "{log_dir}" -f "{log_dir}"

for algo in algos_cuda:
  print('Training:', algo)
  config = f'research_config/{algo}.yml'

  !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
    --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
    --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
    --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 \
    --hyperparams n_timesteps:{n_timestep} env_wrapper:"{wrapper}"\
    --device cuda
  # forward_velocity_weight:5.0 ctrl_cost_weight:1e-3
  !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n {max_episode_steps} \
    --load-best -o "{log_dir}" -f "{log_dir}"



/content/rl-zoo
Training: ppo
2025-08-31 22:46:08.409193: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756680368.428963    9013 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756680368.434945    9013 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756680368.450432    9013 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756680368.450460    9013 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756680368.450463    9013 computation_placer.